In [1]:
import pandas as pd
from bs4 import BeautifulSoup
import numpy as np

In [2]:
with open("SEPTUAGINT.xml") as f:
    sept_soup = BeautifulSoup(f.read())

with open("TISCHENDORF.xml") as file:
    tisch_soup = BeautifulSoup(file)


rm_punct_tbl = str.maketrans("", "", " .·,:;!()«»-·")
sept = pd.DataFrame(
    [
        (
            str(e.text).translate(rm_punct_tbl).lower(),
            e.get("str", "G").removeprefix("G"),
            e.find_parent("vers")["vnumber"],
            e.find_parent("chapter")["cnumber"],
            e.find_parent("biblebook")["bnumber"],
            e["rmac"],
        )
        for e in sept_soup.find_all("gr")
    ],
    columns=["text", "str", "verse", "chapter", "book", "rmac"],
)
tisch = pd.DataFrame(
    [
        (
            str(e.text).translate(rm_punct_tbl).lower(),
            e.get("str"),
            e.find_parent("vers")["vnumber"],
            e.find_parent("chapter")["cnumber"],
            e.find_parent("biblebook")["bnumber"],
            e["rmac"],
        )
        for e in tisch_soup.find_all("gr")
    ],
    columns=["text", "str", "verse", "chapter", "book", "rmac"],
)

In [3]:
sept["verse"] = sept["verse"].astype(int)
sept["chapter"] = sept["chapter"].astype(int)
sept["book"] = sept["book"].astype(int)
# include code to parse rmac?

In [4]:
tisch["verse"] = tisch["verse"].astype(int)
tisch["chapter"] = tisch["chapter"].astype(int)
tisch["book"] = tisch["book"].astype(int)

## Reconciling missing strongs

In [5]:
# words without strongs
nameless = pd.concat(
    [tisch[~tisch["str"].astype(bool)], sept[~sept["str"].astype(bool)]]
)

# unique
no_str_words = pd.Series(nameless["text"].unique())


new_strongs = pd.Series(
    "C" + np.arange(len(no_str_words)).astype(str), index=no_str_words
)

In [6]:
tisch_needs_new = ~tisch["str"].astype(
    bool
)  # if null or "" sets to False, anything else -> True.
tisch.loc[tisch_needs_new, "str"] = new_strongs.loc[
    tisch.loc[tisch_needs_new, "text"]
].values

sept_needs_new = ~sept["str"].astype(bool)
sept.loc[sept_needs_new, "str"] = new_strongs.loc[
    sept.loc[sept_needs_new, "text"]
].values

In [ ]:
import pickle
with open('./pickles/tisch.pickle', 'wb') as f:
    pickle.dump(tisch, f)
with open('./pickles/sept.pickle', 'wb') as f:
    pickle.dump(sept, f)